In [ ]:
import numpy as np
from dataclasses import dataclass
import matplotlib.pyplot as plt
import scipy.optimize


In [ ]:
@dataclass
class Parameters:
    Lx: int
    Ly: int
    Lz: int
    a: float
    sigma3: float
    m3sq: float
    g3: float
    lambda3: float
    rng: np.random.Generator

In [ ]:
def potential(phi: float, p: Parameters):
    return p.sigma3*phi + (1.0/2.0)*p.m3sq*phi**2 + (1.0/6.0)*p.g3*phi**3 + (1.0/24.0)*p.lambda3*phi**4

def laplacian(lattice: np.ndarray, x: int, y: int, z: int, p: Parameters):
    total = 0.0

    total += lattice[(x + 1) % p.Lx][y][z]
    total += lattice[x][(y + 1) % p.Ly][z]
    total += lattice[x][y][(z + 1) % p.Lz]
    total += lattice[(x + p.Lx - 1) % p.Lx][y][z]
    total += lattice[x][(y + p.Ly - 1) % p.Ly][z]
    total += lattice[x][y][(z + p.Lz - 1) % p.Lz]

    total -= 6.0 * lattice[x][y][z]

    return total

def laplacian_contribution(lattice: np.ndarray, x: int, y: int, z: int, p: Parameters):
    total = 0.0
    
    total += 2.0 * lattice[(x + 1) % p.Lx][y][z]
    total += 2.0 * lattice[x][(y + 1) % p.Ly][z]
    total += 2.0 * lattice[x][y][(z + 1) % p.Lz]
    total += 2.0 * lattice[(x + p.Lx - 1) % p.Lx][y][z]
    total += 2.0 * lattice[x][(y + p.Ly - 1) % p.Ly][z]
    total += 2.0 * lattice[x][y][(z + p.Lz - 1) % p.Lz]

    total -= 6.0 * lattice[x][y][z]

    return total

def lattice_contribution(lattice: np.ndarray, x: int, y: int, z: int, p: Parameters):
    total = 0.0

    total += (-1.0/(2.0*p.a**2))*lattice[x][y][z]*laplacian_contribution(lattice, x, y, z, p)
    total += potential(lattice[x][y][z], p)

    return total

def lattice_action(lattice: np.ndarray, p: Parameters):
    total = 0.0

    for x in range(p.Lx):
        for y in range(p.Ly):
            for z in range(p.Lz):                
                total += (-1.0/(2.0*p.a**2))*lattice[x][y][z]*laplacian(lattice, x, y, z, p)
                total += potential(lattice[x][y][z], p)


    return total


In [ ]:
def init_lattice_cold(p: Parameters):
    lattice = np.zeros(shape=(p.Lx,p.Ly,p.Lz), dtype=float)
    return lattice

def init_lattice_warm(p: Parameters, seed=1):

    rng = np.random.default_rng(seed=seed)
    lattice = rng.uniform(-1, 1, size=(p.Lx,p.Ly,p.Lz))
    return lattice

In [ ]:
def metropolis_update(lattice: np.ndarray, x: int, y: int, z: int, p: Parameters):
    original = lattice[x][y][z]

    before = lattice_contribution(lattice, x,y,z, p)  

    lattice[x][y][z] += p.rng.uniform(-1, 1)

    after = lattice_contribution(lattice, x,y,z, p)

    if after < before or np.exp(-1.0*(after-before)) > p.rng.uniform(0,1):
        # print(after-before, 'accept')
        return 1, (after-before)

    # print(after-before, 'reject')
    lattice[x][y][z] = original
    return 0, 0.0
    
def overrelax_update(lattice: np.ndarray, x: int, y: int, z: int, p: Parameters):

    phi_0 = lattice[x][y][z]

    alpha = (1.0/24.0)*p.lambda3
    beta = (1.0/6.0)*p.g3
    gamma = (-1.0/(2.0*p.a**2)) * -6.0 + (1.0/2.0)*p.m3sq
    delta = 2.0 * (-1.0/(2.0*p.a**2)) * (laplacian(lattice, x, y, z, p) + 6.0 * phi_0) + p.sigma3

    
    #   this is equivalent to solving
    #        alpha*(phi-phi0)*(phi**3 + a*phi**2 + b*phi + d) = 0
    #    where
    #        alpha = phi0
    #        beta = b/a + phi0**2
    #        gamma = d/a + alpha*beta
   
    a = phi_0 + beta / alpha
    b = phi_0 * a + gamma / alpha
    d = phi_0 * b + delta / alpha

  
    #    solving
    #        phi**3 + a*phi**2 + b*phi + d = 0

    def poly(x):
        return x**3 + a*x**2 + b*x + d
    #X = np.linspace(-10,10,500)
    #plt.plot(X, poly(X))
    roots = scipy.optimize.fsolve(poly, 0.1)

    # print('Roots:', roots)

    phi = roots[0]

    if len(roots) == 3:
        prob = p.rng.uniform(0, 1)

        if prob > 2.0/3.0:
            phi = roots[2]
        elif prob > 1.0/3.0:
            phi = roots[1]

    measure = phi_0 * (4.0 * alpha * phi_0 * phi_0 + 3.0 * beta * phi_0 + 2.0 * gamma) + delta
    measure /= phi * (4.0 * alpha * phi * phi + 3.0 * beta * phi + 2.0 * gamma) + delta
    measure = np.log(np.abs(measure))

    lattice[x][y][z] = phi

    if measure > np.log(1.0) or measure > np.log(p.rng.uniform(0, 1)):
        return 1
    
    lattice[x][y][z] = phi_0
    return 0

In [ ]:
def sweep(lattice: np.ndarray, p: Parameters):
    total_change = 0.0
 
    updated = 0
    accepted = 0
    before = lattice_action(lattice, p)

    for evenodd in (0,1):
        for x in range(p.Lx):
            for y in range(p.Ly):
                for z in range(p.Lz):
                    if (x + y + z) % 2 == evenodd:
                        accept, change = metropolis_update(lattice, x, y, z, p)
                        total_change += change
                        updated += 1
                        accepted += accept
    
    after = lattice_action(lattice, p)
    # print(before, after, total_change, after-before, (after-before)-total_change)
    return updated, accepted

def sweep_overrelax(lattice: np.ndarray, p: Parameters):
    total_change = 0.0
 
    updated = 0
    accepted = 0

    for evenodd in (0,1):
        for x in range(p.Lx):
            for y in range(p.Ly):
                for z in range(p.Lz):
                    if (x + y + z) % 2 == evenodd:
                        accept= overrelax_update(lattice, x, y, z, p)
                        updated += 1
                        accepted += accept
    
    after = lattice_action(lattice, p)
    # print(before, after, total_change, after-before, (after-before)-total_change)
    return updated, accepted

In [ ]:
def avg_phi(lattice: np.ndarray, p: Parameters):
    total = 0.0

    for x in range(p.Lx):
        for y in range(p.Ly):
            for z in range(p.Lz):
                total += lattice[x][y][z]

    return total/(p.Lx*p.Ly*p.Lz)

def avg_phi_sq(lattice: np.ndarray, p: Parameters):
    total = 0.0

    for x in range(p.Lx):
        for y in range(p.Ly):
            for z in range(p.Lz):
                total += lattice[x][y][z]**2

    return total/(p.Lx*p.Ly*p.Lz)

In [ ]:
def run(sweeps: int, lattice: np.ndarray, p: Parameters):

    total_updated = 0
    total_accepted = 0

    total_overrelax_updated = 0
    total_overrelax_accepted = 0
    results = []

    for i in range(sweeps):
        for j in range(10):
            updated, accepted = sweep(lattice, p)
            total_updated += updated
            total_accepted += accepted

        updated, accepted = sweep_overrelax(lattice, p)
        total_overrelax_updated += updated
        total_overrelax_accepted += accepted        
        results.append((avg_phi(lattice, p),avg_phi_sq(lattice,p)))
        
    print('Metropolis: %d sites updated, %d accepted, %g rate' % (total_updated, total_accepted, float(total_accepted)/float(total_updated)))
    print('Overrelax: %d sites updated, %d accepted, %g rate' % (total_overrelax_updated, total_overrelax_accepted, float(total_overrelax_accepted)/float(total_overrelax_updated)))
    return results


In [ ]:
# p = Parameters(Lx=6, Ly=6, Lz=6, a=2, sigma3=0.025, m3sq=-0.035, g3=-0.3, lambda3=1, rng=np.random.default_rng(seed=2))
p = Parameters(Lx=6, Ly=6, Lz=6, a=1, sigma3=0.005, m3sq=-0.7, g3=0, lambda3=5, rng=np.random.default_rng(seed=5))

lattice = init_lattice_warm(p,seed=2)

In [ ]:
x = np.linspace(-1.5,1.5,50)
plt.plot(x, potential(x, p))

In [ ]:
measurements = []

In [ ]:
measurements += run(2000, lattice, p)

In [ ]:
print(len(measurements))
phi = np.array(measurements)[:,0]
# phisq = np.array(measurements)[:,1]

plt.scatter(range(len(phi)),phi)
# plt.scatter(range(len(phisq)),phisq)

In [ ]:
(counts, edges, patches) = plt.hist(phi, bins=50)
#(counts, edges, patches) = plt.hist(phisq, bins=50)